# 03-combined-indicators

See [project guide](../../README.md) and [data requirements](../../data/README.md) before execution. Workspace: `data/mobility/`. External inputs are not included. Run cells in order; model fitting and network collection are not run during repository checks.


In [ ]:
from pathlib import Path
import sys
import os
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'project_paths.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT))
from project_paths import workspace
os.chdir(workspace('mobility'))


In [ ]:
%pprint
import os
os.environ["KMP_DUPLICATE_LIB_OK"]  =  "TRUE"

In [ ]:
import numpy as np 
import pandas as pd
import os
import time
import datetime
import requests
import csv
import json
import re
import threading

import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
from tqdm import tqdm

%config InlineBackend.figure_format = 'retina'
import warnings
# Keep warnings visible when checking the research environment.

# Global Function

In [ ]:
from analysis_utils import formalize_fip
def getCounties():
    # get a dictionary that maps a formalized fip (len == 5 str) to the corresponding county
    d = {}
    r = requests.get("https://www2.census.gov/geo/docs/reference/codes/files/national_county.txt", timeout=30)
    r.raise_for_status()
    reader = csv.reader(r.text.splitlines(), delimiter=',')    
    for line in reader:
        d[line[1] + line[2]] = line[3].replace(" County","")
    
    extra_fip2county  = {'46102': 'Oglala Lakota', '02158': 'Kusilvak Census Area'}
    d.update(extra_fip2county)       
        
    return d
from analysis_utils import transform_mobilitydata
def compare_dtspp(fip_tuple, start_day='2020-03-01', end_day='2020-09-01', display_policy=False, plot_pic=False):
    fips = [formalize_fip(fip) for fip in fip_tuple]
    if display_policy:
        policies = poly if 'fips' in poly.columns else poly.reset_index()
        display(policies.loc[policies.fips.isin(fips)])
    result = transform_mobilitydata(dtspp.loc[dtspp.fips.isin(fips)],
                                   start_day=start_day, end_day=end_day)
    if plot_pic:
        result.plot(figsize=(15, 6))
    return result


# Import Data

In [ ]:
# Locate folder
os.chdir(workspace('mobility'))     

# dtspp mobility
dtspp = pd.read_csv('Delta TSPP/DTSPP_US_Mobility_formalized.csv',parse_dates=[4])
dtspp.fips = dtspp.fips.apply(formalize_fip)
dtspp_trans = transform_mobilitydata(dtspp)
dtspp.date = pd.to_datetime(dtspp.date)

# policy
poly = pd.read_csv('Local-Policy-Responses-formalized-matched-localonly.csv')
poly.fips = poly.fips.apply(formalize_fip)
poly = poly.set_index('fips', drop=True)


# consumption
csp_byday = pd.read_csv('trends/google-trend-Coronavirus_disease_2019_byday_sorted.csv',index_col=0)    # compare by day
csp_byreg = pd.read_csv('trends/google-trend-Coronavirus_disease_2019_byregion_sorted.csv',index_col=0) # compare by region

# metro to county
metro2county = pd.read_csv('trends/googel-trend-metro2county-clean-version.csv',index_col=0)

# us covid count
# covid = pd.read_csv('us-covidcase-counties-2020.csv')
covid = pd.read_csv('matched-us-covidcase-counties-2020.csv')
covid.fips = covid.fips.apply(formalize_fip)
covid.date = pd.to_datetime(covid.date)

## Covid Count Clean & Policy Data formalized

In [ ]:
covid['fips']=covid.fips.apply(formalize_fip)
covid = covid[~covid.fips.isna()]
covid.date = pd.to_datetime(covid.date)

In [ ]:
def invers_cumsum(lst):
    formerlst = lst[:len(lst)-1].copy()
    formerlst.insert(0,0)
    result = list(np.array(lst)-np.array(formerlst))
    return result
def add_miss(df):
    frame = pd.DataFrame(index=pd.date_range(start='2020-01-04', end='2020-12-28'), columns=df.columns[1:])
    df = df.set_index('date').reindex_like(frame)
    df.fips = df.fips.value_counts().index[0]
    df.cases.loc[pd.to_datetime('2020-01-04'):df.cases.first_valid_index()- pd.Timedelta(days=1)] = 0
    df.deaths.loc[pd.to_datetime('2020-01-04'):df.deaths.first_valid_index()- pd.Timedelta(days=1)] = 0
    df.cases.loc[pd.to_datetime('2020-01-04'):df.cases.last_valid_index()] = df.cases.loc[pd.to_datetime('2020-01-04'):df.cases.last_valid_index()].interpolate(method='linear', limit_direction='forward', limit=5,axis=0)
    df.deaths.loc[pd.to_datetime('2020-01-04'):df.deaths.last_valid_index()] = df.deaths.loc[pd.to_datetime('2020-01-04'):df.deaths.last_valid_index()].interpolate(method='linear', limit_direction='forward', limit=5,axis=0)
    df['new_cases'] = invers_cumsum(list(df.cases))
    df['new_deaths'] = invers_cumsum(list(df.deaths))
    df = df.reset_index().rename(columns={'index':'date'})
    return df

In [ ]:
covid_clean = covid.groupby('fips').apply(add_miss)

In [ ]:
# covid_clean.reset_index(drop=True).drop(columns={'county','state'}).to_csv('matched-us-covidcase-counties-2020.csv', index=False)

In [ ]:
def formalized_policy(df):
    try:
        result = df[df.cityname.isna()].iloc[0,:][['state', 'countyname', 'fips', 'stsipstart','stsipend', 'localsipstart', 'localsipend', 'stbusclose', 'localbusclose','stbusopen','localbusopen', 'stresclose',
           'stresopen', 'localresclose', 'localresopen']]
    except:
        result = pd.Series(df.iloc[0,:][['state', 'countyname', 'fips', 'stsipstart','stsipend', 'localsipstart', 'localsipend', 'stbusclose', 'localbusclose','stbusopen','localbusopen', 'stresclose',
           'stresopen', 'localresclose', 'localresopen']])

        for i in ['localsipstart', 'localbusclose', 'localresclose']:
            result[i] = list(df[i].sort_values())[0]

        for j in ['localsipend', 'localbusopen', 'localresopen']:
            result[j] = list(df[j].sort_values())[-1]
    return result
def update_poly(state_, county_, append_date, feature, cover=False):
    if cover:
            if county_ == '0':
                if append_date == '0':
                    poly.loc[poly[(poly.state == state_) ].index, feature ] = np.nan
                else:
                    poly.loc[poly[(poly.state == state_) ].index, feature ] = pd.to_datetime(append_date).date()
                display(poly[poly.state == state_])
            else:
                if append_date == '0':
                    poly.loc[poly[(poly.state == state_) & (poly.countyname == county_) ].index, feature ] = np.nan
                else:
                    poly.loc[poly[(poly.state == state_) & (poly.countyname == county_) ].index, feature ] = pd.to_datetime(append_date).date()

                display(poly[(poly.state == state_) & (poly.countyname == county_)])

    else:
        
        if county_ == '0':
            if append_date == '0':
                poly.loc[poly[(poly.state == state_) & (poly[feature].isna())].index, feature ] = np.nan
            else:
                poly.loc[poly[(poly.state == state_) & (poly[feature].isna())].index, feature ] = pd.to_datetime(append_date).date()
            display(poly[poly.state == state_])
        else:
            if append_date == '0':
                poly.loc[poly[(poly.state == state_) & (poly.countyname == county_) & (poly[feature].isna()) ].index, feature ] = np.nan
            else:
                poly.loc[poly[(poly.state == state_) & (poly.countyname == county_) & (poly[feature].isna()) ].index, feature ] = pd.to_datetime(append_date).date()

            display(poly[(poly.state == state_) & (poly.countyname == county_)])

In [ ]:
# Fill only missing local dates, preserving observed local policies.
for local, state in [('localsipstart', 'stsipstart'), ('localsipend', 'stsipend'),
                     ('localbusclose', 'stbusclose'), ('localbusopen', 'stbusopen'),
                     ('localresclose', 'stresclose'), ('localresopen', 'stresopen')]:
    poly[local] = poly[local].fillna(poly[state])
poly = poly.drop(columns=['fips.1', 'stsipstart', 'stsipend', 'stbusclose',
                          'stbusopen', 'stresclose', 'stresopen'], errors='ignore')


In [ ]:
pd.set_option('display.max_rows', None)
display(poly[(poly.state == 'WA')])
pd.set_option('display.max_rows', 10)

In [ ]:
#########################################################################
state_ = 'NC'
poly.loc[poly[(poly.state == state_) & (poly.localbusopen.isna())].index, 'localbusclose'] = np.nan
poly.loc[poly[(poly.state == state_) & (poly.localresopen.isna())].index, 'localresclose'] = np.nan

In [ ]:
state_ = 'NY'
county_ = '0'
append_date = '2020-05-28'
feature = 'localbusopen'

update_poly(state_, county_, append_date, feature, cover=False)

In [ ]:
# poly.to_csv('Local-Policy-Responses-formalized-matched-localonly.csv')

## Match Data

In [ ]:
# Build combined_info in the following cells; no pre-existing output is required.


In [ ]:
dtspp.head()


In [ ]:
if dtspp.duplicated(['fips', 'date']).any() or covid.duplicated(['fips', 'date']).any():
    raise ValueError('Duplicate county/date keys; resolve before matching')
combined_info = dtspp.set_index(['fips','date']).copy()

for feature in ['cases', 'deaths', 'new_cases', 'new_deaths']:
    combined_info[feature] = covid.set_index(['fips','date'])[feature]
    
county_nopoly = set([i[0] for i in combined_info.index])-set(poly.index)
combined_info = combined_info.drop(list(county_nopoly), level=0, axis=0)    


for i in ['btw_sip', 'btw_bus', 'btw_res']:
    combined_info[i] = np.nan

In [ ]:
def poly_dummy(ply):
    result = pd.DataFrame(index=pd.date_range(start='2020-01-04',end='2020-12-28'), 
                      columns=['fips','btw_sip', 'btw_bus', 'btw_res'])
    result.fips = ply.name

    try:
        result.loc[ply.localsipstart:ply.localsipend,'btw_sip'] = 1
    except:
        if isinstance(ply.localsipstart, str):
            result.loc[ply.localsipstart:,'btw_sip'] = 1



    try:
        result.loc[ply.localbusclose:ply.localbusopen,'btw_bus'] = 1
    except:
        if isinstance(ply.localbusclose, str):
            result.loc[ply.localbusclose:,'btw_bus'] = 1

    try:
        result.loc[ply.localresclose:ply.localresopen,'btw_res'] = 1
    except:
        if isinstance(ply.localresclose, str):
            result.loc[ply.localresclose:,'btw_res'] = 1

    result.loc[result[result.btw_sip.isna()].index, 'btw_sip'] = 0        
    result.loc[result[result.btw_bus.isna()].index, 'btw_bus'] = 0
    result.loc[result[result.btw_res.isna()].index, 'btw_res'] = 0
    
    result = result.reset_index().rename(columns={'index':'date'}).set_index(['fips','date'])
    
    return result

In [ ]:
# Add policy
frames = []

for cty in tqdm(poly.index):
    frames.append(poly_dummy(poly.loc[cty,:]))

poly_dummy_frame = pd.concat(frames)
combined_info[['btw_sip', 'btw_bus', 'btw_res']] = poly_dummy_frame
combined_info = combined_info.drop(columns=['admin_level']).rename(columns={'admin2':'county'})

In [ ]:
combined_info.to_csv('combined_info.csv')
# Add the Descartes Labs measures consumed by the visualization notebook.
dl = pd.read_csv('Descartes Labs/DL_US_Mobility_formalized.csv')
dl['date'] = pd.to_datetime(dl['date'], errors='raise')
dl['fips'] = dl['fips'].map(formalize_fip)
combined_with_dl = combined_info.reset_index().merge(
    dl[['fips', 'date', 'm50', 'm50_pct_change']],
    on=['fips', 'date'], how='left', validate='one_to_one')
combined_with_dl.to_csv('combined_info_with_dl.csv', index=False)


# Metro-Wise Mobility: By mean

In [ ]:
metro2fip = metro2county.county_code.apply(lambda x: tuple(x.split(', ')))
metro_dtspp = pd.DataFrame(columns=metro2fip.index, index=dtspp_trans.index)

In [ ]:
%%time
for metro in tqdm(metro2fip.index):
    metro_dtspp[metro] = compare_dtspp(metro2fip[metro]).mean(axis=1)
metro_dtspp = metro_dtspp.dropna(axis=1)    

In [ ]:
metro_dtspp

In [ ]:
metro_dtspp.plot(figsize=(20,10), legend=False);

In [ ]:
# metro_dtspp.to_csv('./result/metro-dtspp-mobility.csv')